# 01 — EDA: M5 FOODS weekly demand

Exploratory analysis backing the Phase 4 pipeline decisions (roadmap Phase 4, step 2).
Uses the *implemented* pipeline (`demand_vae.data`) — the notebook explores, the package implements.

**Questions this notebook answers, with the design decision each supports:**
1. Is weekly FOODS demand over-dispersed and better fit by a Negative Binomial than Poisson/Gaussian? → decoder likelihood (design doc §10.4).
2. How intermittent are the weekly series? → weekly aggregation choice (assumption A2).
3. Is there week-of-year seasonality worth conditioning on? → sin/cos week-of-year context feature.
4. What do representative series look like across the temporal splits? → sanity for the split protocol.

Requires the M5 data in `data/raw/` (`python scripts/download_data.py`). Full run ≈ 15 s.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from demand_vae.config import load_config
from demand_vae.data.pipeline import aggregate_weekly, load_raw_m5, temporal_split
from demand_vae.data.validation import zero_fraction_report

REPO = Path.cwd() if (Path.cwd() / "configs").exists() else Path.cwd().parent
FIGURES = REPO / "figures"

config = load_config(REPO / "configs" / "default.yaml")
panel = aggregate_weekly(load_raw_m5(REPO / "data"), config)
spans = temporal_split(panel, config)
train_weeks = np.arange(spans["train"][0], spans["train"][1] + 1)
print(
    f"{panel.n_series} series x {panel.n_weeks} weeks "
    f"({panel.n_dropped_series} series dropped: no train-period price)"
)
print("spans:", spans)

## 1. Weekly demand histogram with Gaussian / Poisson / NB fits

The single figure that justifies the decoder choice. Fits use **train weeks only** and
method-of-moments parameters; shown for a representative mid-volume series. Gaussian
puts mass on negative demand and misses the right skew; Poisson (variance = mean) is
far too narrow; NB tracks the over-dispersed shape.

In [ ]:
report = zero_fraction_report(panel)
print(
    f"median dispersion (var/mean): {report.dispersion.median():.2f} | "
    f"over-dispersed series: {(report.dispersion > 1).mean():.1%}"
)

# Representative series: median-ish mean demand among low-intermittency series.
candidates = report[(report.zero_frac < 0.05) & (report.n_available_weeks > 200)]
rep_id = candidates.sort_values("mean").iloc[len(candidates) // 2]["series_id"]
row = list(panel.series_ids).index(rep_id)
demand = panel.sales[row, train_weeks][panel.available[row, train_weeks]]
mean, var = demand.mean(), demand.var()

ks = np.arange(0, int(demand.max() * 1.5))
gauss = stats.norm.pdf(ks, mean, demand.std())
poiss = stats.poisson.pmf(ks, mean)
r = mean**2 / max(var - mean, 1e-9)  # MoM: var = mu + mu^2/r
nb = stats.nbinom.pmf(ks, r, r / (r + mean))

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(
    demand,
    bins=np.arange(demand.max() + 2) - 0.5,
    density=True,
    alpha=0.4,
    color="gray",
    label="observed weekly demand (train)",
)
ax.plot(ks, gauss, label="Gaussian fit", lw=2)
ax.plot(ks, poiss, label="Poisson fit", lw=2)
ax.plot(ks, nb, label="Negative Binomial fit", lw=2)
ax.set_title(f"{rep_id}  (mean={mean:.1f}, var/mean={var / mean:.1f})")
ax.set_xlabel("weekly unit sales")
ax.set_ylabel("density")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "demand_hist_fits.png", dpi=150)

## 2. Intermittency: zero fraction per series

Daily M5 series are majority-zero; weekly aggregation is what makes likelihood
estimation workable (assumption A2). This shows how much intermittency remains.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(report.zero_frac, bins=50, color="tab:blue", alpha=0.75)
ax.axvline(
    report.zero_frac.median(), color="k", ls="--", label=f"median = {report.zero_frac.median():.2f}"
)
ax.set_xlabel("fraction of available weeks with zero sales")
ax.set_ylabel("number of series")
ax.set_title("Weekly intermittency across FOODS item-store series")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / "zero_fraction.png", dpi=150)
print(
    f">50% zeros: {(report.zero_frac > 0.5).mean():.1%} of series | "
    f"<10% zeros: {(report.zero_frac < 0.1).mean():.1%}"
)

## 3. Seasonal profile: mean demand by week-of-year

Computed on train weeks only. The systematic week-of-year variation is exactly what
the sin/cos context features hand to the CVAE.

In [ ]:
woy = panel.week_start.isocalendar().week.to_numpy()
avail = panel.available[:, train_weeks]
sales = np.where(avail, panel.sales[:, train_weeks], np.nan)
per_week_mean = np.nanmean(sales, axis=0)  # mean across series per train week
profile = pd.Series(per_week_mean, index=woy[train_weeks]).groupby(level=0).mean()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(profile.index, profile.values, marker="o", ms=3)
ax.set_xlabel("ISO week of year")
ax.set_ylabel("mean weekly unit sales per series")
ax.set_title("Seasonal profile of FOODS demand (train period)")
fig.tight_layout()
fig.savefig(FIGURES / "seasonal_profile.png", dpi=150)

## 4. Representative series across the splits

High / mid / low volume plus one intermittent series, with the pre-registered split
boundaries marked. Vertical lines: end of train (2014-07-25) and end of val (2015-04-24).

In [ ]:
by_volume = report.sort_values("mean")
picks = {
    "high volume": by_volume.iloc[-2]["series_id"],
    "mid volume": by_volume.iloc[len(by_volume) // 2]["series_id"],
    "low volume": by_volume.iloc[len(by_volume) // 10]["series_id"],
    "intermittent": report.iloc[10]["series_id"],  # report is sorted by zero_frac desc
}
train_end = pd.Timestamp(config.data.split.train_end)
val_end = pd.Timestamp(config.data.split.val_end)

fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
ids = list(panel.series_ids)
for ax, (label, sid) in zip(axes, picks.items()):
    r = ids.index(sid)
    y = np.where(panel.available[r], panel.sales[r], np.nan)
    ax.plot(panel.week_start, y, lw=0.9)
    for boundary in (train_end, val_end):
        ax.axvline(boundary, color="k", ls="--", lw=0.8)
    ax.set_ylabel("units/wk")
    ax.set_title(f"{label}: {sid}", fontsize=9, loc="left")
axes[-1].set_xlabel("week")
fig.suptitle("Representative FOODS series (dashed: train/val and val/test boundaries)")
fig.tight_layout()
fig.savefig(FIGURES / "sample_series.png", dpi=150)

## Takeaways

- **Over-dispersion is near-universal** (median var/mean ≈ 4; >99% of series above 1):
  Poisson is structurally too narrow, Gaussian leaks mass below zero — the Negative
  Binomial decoder is the empirically supported choice (RQ1's premise).
- **Weekly aggregation tames intermittency** but does not remove it: a long tail of
  series remains majority-zero, which the evaluation must not silently drop.
- **Week-of-year seasonality is real** and systematic — context worth conditioning on.
- Series volumes span orders of magnitude, supporting per-series baselines (Phase 5)
  as serious competition for the pooled deep model.